# Ejemplos de uso de CrewAI


In [2]:
pip install crewai==0.98.0

^C
Note: you may need to restart the kernel to use updated packages.


In [1]:
from crewai import Agent, Task, Crew, Process, LLM
from langchain_openai import AzureChatOpenAI
import os 
from dotenv import load_dotenv

load_dotenv('../.env')
llm = LLM(model = "azure/gpt-35-turbo", api_version = '2024-08-01-preview')


# Crear agentes
researcher = Agent(
    role='Investigador',
    goal='Investigar nuevas tecnologías',
    backstory='Experto en análisis tecnológico',
    llm=llm
)

writer = Agent(
    role='Escritor',
    goal='Crear contenido técnico',
    backstory='Escritor técnico especializado',
    llm=llm
)


# Definir tareas
research_task = Task(
    description='Investigar tendencias en IA',
    agent=researcher,
    expected_output='Informe con hallazgos y conclusiones'
)

writing_task = Task(
    description='Escribir reporte sobre hallazgos',
    agent=writer,
    expected_output='Reporte técnico sobre IA'
)


# Crear y ejecutar crew
crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, writing_task],
    process=Process.sequential
)


result = crew.kickoff()

LLM value is already an LLM object
LLM value is already an LLM object


In [2]:
result

CrewOutput(raw="Introduction:\n\nArtificial intelligence (AI) is a vast field that encompasses multiple subfields, including machine learning, deep learning, natural language processing (NLP), and computer vision. In recent years, AI has gained significant attention due to its potential to revolutionize different industries' operations and processes. This report will analyze the ongoing trends in AI, which include deep learning, NLP, virtual assistants, and machine learning applications.\n\nDeep Learning:\n\nDeep learning is a subfield of AI that involves training systems to learn from examples and recognize patterns in data. Deep neural networks simulate the human brain's behavior, making them ideal for image and speech recognition. Deep learning is used in various applications, such as facial recognition, object detection, speech recognition, and autonomous vehicles.\n\nNLP:\n\nNatural language processing (NLP) is a field of AI that enables computers to understand, interpret, and res

In [4]:
print(result.raw)

Introduction:

Artificial intelligence (AI) is a vast field that encompasses multiple subfields, including machine learning, deep learning, natural language processing (NLP), and computer vision. In recent years, AI has gained significant attention due to its potential to revolutionize different industries' operations and processes. This report will analyze the ongoing trends in AI, which include deep learning, NLP, virtual assistants, and machine learning applications.

Deep Learning:

Deep learning is a subfield of AI that involves training systems to learn from examples and recognize patterns in data. Deep neural networks simulate the human brain's behavior, making them ideal for image and speech recognition. Deep learning is used in various applications, such as facial recognition, object detection, speech recognition, and autonomous vehicles.

NLP:

Natural language processing (NLP) is a field of AI that enables computers to understand, interpret, and respond to human language. NL

In [22]:
from crewai.tools import BaseTool
from pydantic import BaseModel, Field
from typing import Dict
import json
import requests
import pandas as pd

class ForecastToolInput(BaseModel):
    """Input schema for ForecastTool."""
    args: Dict = Field(..., description="Input arguments for forecasting; in the format of {{'target_column': 'name_of_target_column', 'prediction_length': 10}}")

class ForecastTool(BaseTool):
    """Herramienta para generar pronósticos de series temporales."""
    
    name: str = "forecast_tool"
    description: str = """Útil para generar pronósticos de series temporales.
    Requiere un JSON con el siguiente formato:
    {
        "target_column": "nombre_columna",  # Columna objetivo para el pronóstico
        "prediction_length": 30             # Número de períodos a predecir (opcional, default 30)
    }
    Ejemplo de uso: {"target_column": "ventas", "prediction_length": 30}
    Devuelve un DataFrame con las predicciones y los intervalos de confianza."""
    args_schema: type[BaseModel] = ForecastToolInput
    
    def __init__(self, chronos_endpoint: str):
        super().__init__()
        self._chronos_endpoint = chronos_endpoint
        # current_df = None
        self._current_df = None
    
    def _run(self, args: dict) -> Dict:
        """Ejecuta el pronóstico basado en los argumentos proporcionados de forma síncrona"""
        try:
            # Parsear los argumentos
            #params = json.loads(args)
            args = args.get('args')
            target_column = args.get('target_column')
            prediction_length = args.get('prediction_length', 30)
            
            if not target_column:
                return {"error": "target_column is required"}

            if self._current_df is None:
                return {"error": "No data loaded"}

            try:
                # Prepare CSV data
                csv_data = self._current_df.to_csv(index=False)
                
                # Prepare the request
                files = {
                    'file': ('../assets/orders_example.csv', csv_data, 'text/csv')
                }
                data = {
                    'prediction_length': prediction_length,
                    'target_column': target_column
                }
                response = requests.post(self._chronos_endpoint, data=data, files=files)
                if response.status_code == 200:
                    last_forecast = response.json()

                    # Extend the target column with forecast values
                    extended_values = last_forecast['actual'] + last_forecast['forecast']
                    lower_quantile = [None for _ in range(len(last_forecast['actual']))] + last_forecast['lower_quantile']
                    upper_quantile = [None for _ in range(len(last_forecast['actual']))] + last_forecast['upper_quantile']
                    future_assigned = [False] * len(last_forecast['actual']) + [True] * len(last_forecast['forecast'])

                    # Update the DataFrame
                    # Get the periodicity of the first column
                    dates = pd.to_datetime(self._current_df[self._current_df.columns.tolist()[0]])
                    freq = pd.infer_freq(dates)

                    # Generate future dates
                    future_dates = pd.date_range(start=dates.iloc[-1], periods=prediction_length + 1, freq=freq)[1:]

                    # Extend the DataFrame with future dates
                    self._current_df = pd.concat([
                        self._current_df, 
                        pd.DataFrame({self._current_df.columns.tolist()[0]: future_dates})
                    ]).reset_index(drop=True)
                    self._current_df[target_column] = pd.Series(extended_values[:len(self._current_df)])
                    self._current_df['future_assigned_column'] = pd.Series(future_assigned[:len(self._current_df)])
                    self._current_df['lower_quantile'] = pd.Series(lower_quantile[:len(self._current_df)])
                    self._current_df['upper_quantile'] = pd.Series(upper_quantile[:len(self._current_df)])
                    return self._current_df
                else:
                    return {"error": f"Forecast error: {response.text}"}
            except Exception as e:
                return {"error": f"Error generating forecast: {str(e)}"}
                
        except Exception as e:
            return {"error": f"Error in forecast tool: {str(e)}"}
    
    def _arun(self, args: str):
        raise NotImplementedError("Async not implemented")


In [13]:
from crewai.tools.structured_tool import CrewStructuredTool
from pydantic import BaseModel

class APICallInput(BaseModel):
    """Input schema for APICallTool."""
    chronos_endpoint: str = "http://localhost:8080/forecast"
    parameters: str = '{"target_column": "ventas", "prediction_length": 30}'

ModuleNotFoundError: No module named 'crewai.tools.structured_tooll'

In [23]:
from crewai import Agent, Task, Crew, Process, LLM

forecast_tool = ForecastTool(chronos_endpoint='http://localhost:8000/forecast')
forecast_tool._current_df = pd.read_csv('../assets/orders_example.csv')

Forecaster = Agent(
    role='Pronosticador',
    goal='Enviar solicitudes y recibir pronósticos de series temporales',
    backstory='Paso intermedio para la generación de pronósticos',
    llm=llm,
    tools=[forecast_tool],
    allow_delegation=False,
    verbose=True
)




LLM value is already an LLM object


In [24]:
Forecaster_task = Task(
    description='Solicitar pronóstico de series temporales. Hay que pronosticar la columna "ventas" para los próximos 30 períodos',
    agent=Forecaster,
    expected_output='DataFrame con las predicciones y los intervalos de confianza'
)

crew = Crew(
    agents=[Forecaster],
    tasks=[Forecaster_task],
    process=Process.sequential
)

result = crew.kickoff()
print(result.raw)

Overriding of current TracerProvider is not allowed


# Agent: Pronosticador
## Task: Solicitar pronóstico de series temporales. Hay que pronosticar la columna "ventas" para los próximos 30 períodos


# Agent: Pronosticador
## Thought: Para solicitar el pronóstico de la columna "ventas" usando la herramienta "forecast_tool", necesito un JSON con el nombre de la columna objetivo y el número de períodos que se desean predecir.
## Using tool: forecast_tool
## Tool Input: 
"{\"args\": {\"description\": \"Input arguments for forecasting; in the format of {{'target_column': 'name_of_target_column', 'prediction_length': 10}}\", \"type\": \"dict\", \"value\": {\"target_column\": \"ventas\", \"prediction_length\": 30}}}"
## Tool Output: 
Error: the Action Input is not a valid key, value dictionary.


# Agent: Pronosticador
## Thought: Got it! Let me rephrase my previous comment:
Thought: Para solicitar el pronóstico de la columna "ventas" necesito usar la herramienta "forecast_tool" y proporcionar un JSON con el nombre de la columna objetivo "vent